In [2]:
# Import pandas for working with our time-series data.
import pandas as pd

# Load the weekly demand dataset created on Day 1.
df = pd.read_parquet("../artifacts/weekly.parquet")

# Sort by product and date so the time-series is in the correct order.
df = df.sort_values(["item", "date"]).reset_index(drop=True)

# Display the first few rows to confirm everything loaded correctly.
df.head()

,item,date,demand
0,1,2013-01-13,863
1,1,2013-01-20,865
2,1,2013-01-27,799
3,1,2013-02-03,954
4,1,2013-02-10,935


In [3]:
# Create a copy so we can safely add forecasting features.
features_df = df.copy()

# For each product, take the demand from the previous week.
features_df["lag_1"] = (
    features_df
    .groupby("item")["demand"]
    .shift(1)
)

# Display the demand and lag feature to check the result.
features_df[["item", "date", "demand", "lag_1"]].head(10)

,item,date,demand,lag_1
0,1,2013-01-13,863,NaN
1,1,2013-01-20,865,863.0
2,1,2013-01-27,799,865.0
3,1,2013-02-03,954,799.0
4,1,2013-02-10,935,954.0
5,1,2013-02-17,947,935.0
6,1,2013-02-24,961,947.0
7,1,2013-03-03,1096,961.0
8,1,2013-03-10,1182,1096.0
9,1,2013-03-17,1134,1182.0


In [4]:
# Create a 4-week lag for each product.
# This tells the model what demand was about one month ago.
features_df["lag_4"] = (
    features_df
    .groupby("item")["demand"]
    .shift(4)
)

# Create a 13-week lag for each product.
# This captures a roughly quarterly demand pattern.
features_df["lag_13"] = (
    features_df
    .groupby("item")["demand"]
    .shift(13)
)

# Create a 52-week lag for each product.
# This captures yearly seasonality.
features_df["lag_52"] = (
    features_df
    .groupby("item")["demand"]
    .shift(52)
)

# Display the new features to check that they were created correctly.
features_df[
    ["item", "date", "demand", "lag_1", "lag_4", "lag_13", "lag_52"]
].head(60)

,item,date,demand,lag_1,lag_4,lag_13,lag_52
0,1,2013-01-13,863,NaN,NaN,NaN,NaN
1,1,2013-01-20,865,863.0,NaN,NaN,NaN
2,1,2013-01-27,799,865.0,NaN,NaN,NaN
3,1,2013-02-03,954,799.0,NaN,NaN,NaN
4,1,2013-02-10,935,954.0,863.0,NaN,NaN
5,1,2013-02-17,947,935.0,865.0,NaN,NaN
6,1,2013-02-24,961,947.0,799.0,NaN,NaN
7,1,2013-03-03,1096,961.0,954.0,NaN,NaN
8,1,2013-03-10,1182,1096.0,935.0,NaN,NaN
9,1,2013-03-17,1134,1182.0,947.0,NaN,NaN


In [5]:
# Calculate the average demand from the previous 4 weeks.
# shift(1) prevents the current week's demand from being used.
features_df["rolling_mean_4"] = (
    features_df
    .groupby("item")["demand"]
    .transform(lambda x: x.shift(1).rolling(4).mean())
)

# Calculate the average demand from the previous 13 weeks.
# This gives the model a longer-term view of recent demand.
features_df["rolling_mean_13"] = (
    features_df
    .groupby("item")["demand"]
    .transform(lambda x: x.shift(1).rolling(13).mean())
)

# Calculate how much demand varied over the previous 4 weeks.
# A higher value means demand was more volatile.
features_df["rolling_std_4"] = (
    features_df
    .groupby("item")["demand"]
    .transform(lambda x: x.shift(1).rolling(4).std())
)

# Display the new rolling features.
features_df[
    [
        "item",
        "date",
        "demand",
        "rolling_mean_4",
        "rolling_mean_13",
        "rolling_std_4"
    ]
].head(20)

,item,date,demand,rolling_mean_4,rolling_mean_13,rolling_std_4
0,1,2013-01-13,863,NaN,NaN,NaN
1,1,2013-01-20,865,NaN,NaN,NaN
2,1,2013-01-27,799,NaN,NaN,NaN
3,1,2013-02-03,954,NaN,NaN,NaN
4,1,2013-02-10,935,870.25,NaN,63.693930
5,1,2013-02-17,947,888.25,NaN,70.745436
6,1,2013-02-24,961,908.75,NaN,73.586117
7,1,2013-03-03,1096,949.25,NaN,11.086779
8,1,2013-03-10,1182,984.75,NaN,74.923850
9,1,2013-03-17,1134,1046.50,NaN,112.577381


In [6]:
# Extract the month from the date.
features_df["month"] = features_df["date"].dt.month

# Extract the week number within the year.
features_df["week_of_year"] = features_df["date"].dt.isocalendar().week.astype(int)

# Extract the year.
features_df["year"] = features_df["date"].dt.year

# Display the calendar features to check them.
features_df[
    ["date", "month", "week_of_year", "year"]
].head(15)

,date,month,week_of_year,year
0,2013-01-13,1,2,2013
1,2013-01-20,1,3,2013
2,2013-01-27,1,4,2013
3,2013-02-03,2,5,2013
4,2013-02-10,2,6,2013
5,2013-02-17,2,7,2013
6,2013-02-24,2,8,2013
7,2013-03-03,3,9,2013
8,2013-03-10,3,10,2013
9,2013-03-17,3,11,2013


In [7]:
# Remove rows where any of our features are missing.
# These mainly come from the lag and rolling calculations.
features_df = features_df.dropna().reset_index(drop=True)

# Check how many rows are left.
print("Rows after removing NaNs:", len(features_df))

# Confirm that no missing values remain.
print("\nMissing values:")
print(features_df.isna().sum())

Rows after removing NaNs: 4160

Missing values:
item               0
date               0
demand             0
lag_1              0
lag_4              0
lag_13             0
lag_52             0
rolling_mean_4     0
rolling_mean_13    0
rolling_std_4      0
month              0
week_of_year       0
year               0
dtype: int64


In [8]:
# These are the inputs our LightGBM model will use.
feature_cols = [
    "item",
    "lag_1",
    "lag_4",
    "lag_13",
    "lag_52",
    "rolling_mean_4",
    "rolling_mean_13",
    "rolling_std_4",
    "month",
    "week_of_year",
    "year"
]

# Our prediction target.
target_col = "demand"

print("Features:")
print(feature_cols)

Features:
['item', 'lag_1', 'lag_4', 'lag_13', 'lag_52', 'rolling_mean_4', 'rolling_mean_13', 'rolling_std_4', 'month', 'week_of_year', 'year']


In [9]:
# Find the last date in our dataset.
max_date = features_df["date"].max()

# Keep the final 26-week period as our test period.
test_start = max_date - pd.Timedelta(weeks=26)

print("Test starts:", test_start)
print("Dataset ends:", max_date)

Test starts: 2017-07-02 00:00:00
Dataset ends: 2017-12-31 00:00:00


In [10]:
# Use the same test period we defined on Day 1.
train_mask = features_df["date"] < test_start
test_mask = features_df["date"] >= test_start

# Create training features and target.
X_train = features_df.loc[train_mask, feature_cols]
y_train = features_df.loc[train_mask, target_col]

# Create test features and target.
X_test = features_df.loc[test_mask, feature_cols]
y_test = features_df.loc[test_mask, target_col]

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

print("\nTraining period:")
print(features_df.loc[train_mask, "date"].min(),
      "to",
      features_df.loc[train_mask, "date"].max())

print("\nTest period:")
print(features_df.loc[test_mask, "date"].min(),
      "to",
      features_df.loc[test_mask, "date"].max())

Training rows: 3620
Test rows: 540

Training period:
2014-01-12 00:00:00 to 2017-06-25 00:00:00

Test period:
2017-07-02 00:00:00 to 2017-12-31 00:00:00


In [11]:
# Install LightGBM if it is not already installed.
%pip install lightgbm


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
# Import the LightGBM regression model.
from lightgbm import LGBMRegressor

print("LightGBM imported successfully!")

LightGBM imported successfully!


In [13]:
# Install scikit-learn because LightGBM's sklearn interface needs it.
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# | Parameter       |          Yours | Simple meaning                                |
# | --------------- | -------------: | --------------------------------------------- |
# | `objective`     | `"regression"` | What the model is trying to predict           |
# # | `n_estimators`  |          `500` | Number of boosting trees                      |
# | `learning_rate` |         `0.03` | How much each new tree changes the prediction |
# | `num_leaves`    |           `31` | How complex each tree can become              |
# | `random_state`  |           `42` | Keeps results reproducible                    |


In [16]:
# Create our LightGBM regression model.
model = LGBMRegressor(
    objective="regression",
    n_estimators=500,
    learning_rate=0.03,
    num_leaves=31,
    random_state=42
)

print(model)

LGBMRegressor(learning_rate=0.03, n_estimators=500, objective='regression',
              random_state=42)


In [17]:
# Train LightGBM using our historical training data.
model.fit(X_train, y_train)

print("Model training complete!")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000146 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1878
[LightGBM] [Info] Number of data points in the train set: 3620, number of used features: 11
[LightGBM] [Info] Start training from score 3856.538122
Model training complete!


In [18]:
# Use the trained LightGBM model to predict demand for the test period.
predictions = model.predict(X_test)

# Display the first few predictions.
print("First 10 predictions:")
print(predictions[:10])

First 10 predictions:
[2234.81436218 2294.96805806 2182.4499091  2205.54002859 2267.68774664
 1999.59295301 1902.01378465 1890.62255333 1966.11091784 1871.98528382]


In [19]:
# Create a small table comparing actual demand with model predictions.
results = features_df.loc[
    test_mask,
    ["date", "item", "demand"]
].copy()

# Add the LightGBM predictions.
results["prediction"] = predictions

# Calculate the absolute error for each prediction.
results["absolute_error"] = (
    results["demand"] - results["prediction"]
).abs()

# Display the first 10 predictions.
results.head(10)

,date,item,demand,prediction,absolute_error
181,2017-07-02,1,2119,2234.814362,115.814362
182,2017-07-09,1,2227,2294.968058,67.968058
183,2017-07-16,1,2261,2182.449909,78.550091
184,2017-07-23,1,2239,2205.540029,33.459971
185,2017-07-30,1,2247,2267.687747,20.687747
186,2017-08-06,1,1947,1999.592953,52.592953
187,2017-08-13,1,1961,1902.013785,58.986215
188,2017-08-20,1,1926,1890.622553,35.377447
189,2017-08-27,1,1948,1966.110918,18.110918
190,2017-09-03,1,1887,1871.985284,15.014716


In [20]:
# Calculate Weighted Absolute Percentage Error (WAPE).
wape = (
    results["absolute_error"].sum()
    / results["demand"].sum()
)

print(f"LightGBM WAPE: {wape:.2%}")

LightGBM WAPE: 2.09%


In [21]:
# Save the test predictions so we can use them later.
results.to_parquet("../artifacts/lightgbm_predictions.parquet", index=False)

print("Saved LightGBM predictions.")

Saved LightGBM predictions.


In [22]:
import joblib

# Save the trained LightGBM model for later use.
joblib.dump(model, "../artifacts/lightgbm_model.pkl")

print("Saved LightGBM model.")

Saved LightGBM model.


In [ ]:
%pip freeze > ../requirements.txt